# MVAI · DocTags Kaggle 免费 GPU 推理试点（V1.4）

**用途：** 公开基线模型的真实推理验证，而非团队自研模型权重的展示。默认使用 IBM 开源模型 `ibm-granite/granite-docling-258M`，生成 DocTags。正式接入团队自研 checkpoint 前，任何输出都不得声称来自团队自研模型。

**隐私要求：** 默认创建一张不含个人信息的合成英文文档。请勿上传含姓名、签名、证件号、手机号或未授权公文的真实材料。中文能力属实验性质。

**运行流程：** Kaggle 账号登录并完成必要验证 → Settings 中 Accelerator 选择 GPU（T4/P100 等可用型号）→ Internet 按需启用 → Run All。输出保存在 `/kaggle/working/mvai_doctags_pilot/`。本 Notebook 并不启动公开 API、Gradio 服务或公网隧道，不耗用实验室 GPU。

## 1. 安装推理依赖（Kaggle 内运行）
首次运行需要 Internet 下载公开模型权重，且会使用个人 Kaggle GPU 配额。依赖安装失败时请查看 pip 错误，不要继续运行后续单元。

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.53,<5", "accelerate>=1.3,<2", "docling-core>=2.40,<3", "pymupdf>=1.24,<2", "pillow>=10,<13"])

## 2. 运行环境与参数

In [ ]:
from pathlib import Path
import hashlib, json, time, os, platform
import torch
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

MODEL_ID = "ibm-granite/granite-docling-258M"  # 公开基线，非实验室自研 checkpoint
INPUT_PATH = ""  # 可改成 /kaggle/input/.../xxx.png、.jpg 或 .pdf；留空使用合成样例
PDF_PAGE = 0  # 从 0 计数，仅处理一页
MAX_NEW_TOKENS = 3072  # 大文档可能截断，可酌情增加，但更耗显存/时间
OUTPUT_DIR = Path("/kaggle/working/mvai_doctags_pilot")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "没有检测到 Kaggle GPU。请先在 Settings > Accelerator 启用可用 GPU 后重试。"
DEVICE = "cuda"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| 显存 GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("PyTorch:", torch.__version__, "| Python:", platform.python_version(), "| dtype:", DTYPE)
print("来源：公开基线", MODEL_ID)

## 3. 选择文档
默认生成一个不含个人信息的模拟英文报告，保证首次体验无需额外上传。若要测试自己的文件，先通过 Kaggle 的 **Add Input** 将已脱敏图片或 PDF 加入个人 Notebook，再填写 `INPUT_PATH`。仅分析指定 PDF 页。

In [ ]:
def synthetic_document():
    canvas = Image.new("RGB", (1100, 1450), "white")
    d = ImageDraw.Draw(canvas)
    try:
        font_title = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 42)
        font_body = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 25)
        font_small = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 21)
    except OSError:
        font_title = font_body = font_small = ImageFont.load_default()
    d.text((85, 65), "RESEARCH PROJECT SUMMARY", font=font_title, fill="#182f54")
    d.text((85, 145), "Model evaluation on synthetic documents", font=font_body, fill="#253d5d")
    d.line((85, 200, 1015, 200), fill="#7299c8", width=3)
    d.text((85, 250), "1. Objective", font=font_body, fill="#182f54")
    d.text((85, 315), "Compare document parsing methods using public inputs.", font=font_small, fill="#39495e")
    d.text((85, 380), "2. Evaluation overview", font=font_body, fill="#182f54")
    headers = ["Metric", "Baseline", "Proposed"]
    rows = [["Pages", "20", "20"], ["Runtime (s)", "3.5", "2.8"], ["Table score", "0.71", "0.76"]]
    xs = [85, 425, 715, 1015]; y0=465; height=72
    for ri,row in enumerate([headers]+rows):
        y=y0+ri*height
        if ri == 0: d.rectangle((xs[0],y,xs[-1],y+height),fill="#eaf1fb")
        for ci,val in enumerate(row): d.text((xs[ci]+14,y+21),val,font=font_small,fill="#233c60")
    for xx in xs: d.line((xx,y0,xx,y0+height*4),fill="#7f9fbc",width=2)
    for ri in range(5): d.line((xs[0],y0+ri*height,xs[-1],y0+ri*height),fill="#7f9fbc",width=2)
    d.text((85, 835), "3. Conclusion", font=font_body, fill="#182f54")
    d.text((85, 905), "This page is synthetic and contains no personal data.",font=font_small, fill="#39495e")
    out=OUTPUT_DIR/"synthetic_document.png"; canvas.save(out)
    return canvas, out

if INPUT_PATH.strip():
    source=Path(INPUT_PATH.strip())
    if not source.is_file(): raise FileNotFoundError(f"找不到文件: {source}. 请检查 Kaggle Add Input 中的路径。")
    if source.stat().st_size > 20*1024*1024: raise ValueError("演示限制：单文件不能超过 20MB。")
    if source.suffix.lower()==".pdf":
        import fitz
        with fitz.open(str(source)) as pdf:
            if not (0 <= PDF_PAGE < len(pdf)): raise ValueError("PDF_PAGE 超出页码范围。")
            pix=pdf[PDF_PAGE].get_pixmap(matrix=fitz.Matrix(1.6,1.6),alpha=False)
            image=Image.frombytes("RGB",[pix.width,pix.height],pix.samples)
    elif source.suffix.lower() in {".png",".jpg",".jpeg",".webp"}:
        with Image.open(source) as im: image=im.convert("RGB")
    else:
        raise ValueError("仅支持 PNG/JPG/WEBP 图片或 PDF。")
else:
    image, source=synthetic_document()
image.thumbnail((1700,2200),Image.Resampling.LANCZOS)
image_path=OUTPUT_DIR/"input_page.png"; image.save(image_path)
print("输入:",str(source),"| 处理后的页面尺寸:",image.size)
display(image.resize((min(image.width,680),int(image.height*min(image.width,680)/image.width))))

## 4. 加载公开基线模型
首次运行会下载权重；如果 Kaggle 未开启 Internet 或资源不足，模型加载可能失败。可对照 [Granite Docling 官方模型卡](https://huggingface.co/ibm-granite/granite-docling-258M) 排查。

In [ ]:
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText as DoclingModelClass
except ImportError:
    from transformers import AutoModelForVision2Seq as DoclingModelClass

t_load=time.perf_counter()
processor=AutoProcessor.from_pretrained(MODEL_ID)
try:
    model=DoclingModelClass.from_pretrained(MODEL_ID, torch_dtype=DTYPE, attn_implementation="sdpa").to(DEVICE).eval()
except (TypeError, ValueError) as e:
    print("当前模型/Transformers 不支持指定 attention 配置，重试默认实现:",type(e).__name__)
    model=DoclingModelClass.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE).eval()
load_seconds=time.perf_counter()-t_load
print("模型加载完成，耗时(s):",round(load_seconds,2))

## 5. 实际 GPU 推理并生成 DocTags

In [ ]:
messages=[{"role":"user","content":[{"type":"image"},{"type":"text","text":"Convert this page to docling."}]}]
prompt=processor.apply_chat_template(messages,add_generation_prompt=True)
inputs=processor(text=prompt,images=[image],return_tensors="pt").to(DEVICE)
t0=time.perf_counter()
try:
    with torch.inference_mode():
        generated=model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False)
except torch.cuda.OutOfMemoryError as e:
    torch.cuda.empty_cache()
    raise RuntimeError("GPU 显存不足：请减小页面分辨率、降低 MAX_NEW_TOKENS 或重启会话。") from e
infer_seconds=time.perf_counter()-t0
prompt_len=inputs["input_ids"].shape[1]
new_tokens=generated[:,prompt_len:]
doctags=processor.batch_decode(new_tokens,skip_special_tokens=False)[0].strip()
if not doctags: raise RuntimeError("模型没有返回 DocTags 文本，请检查提示词与权重版本。")
print("推理完成：",round(infer_seconds,2),"秒；生成 token 数：",new_tokens.shape[1])
print("DocTags 前 1600 字符（完整结果将在输出文件中）：\n",doctags[:1600])

## 6. 转换、保存和下载结果
优先保留模型直接生成的 `.dt` 文件。DoclingDocument 转换失败时仍提供原始 DocTags，并在 `run_metadata.json` 中记录失败原因。**合成样例完成一次推理是最小验收标准；不代表所有中文文档都解析准确。**

In [ ]:
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
result_md=None
conversion_error=None
try:
    dt_doc=DocTagsDocument.from_doctags_and_image_pairs([doctags],[image])
    doc=DoclingDocument.load_from_doctags(dt_doc,document_name="MVAI DocTags Kaggle pilot")
    result_md=doc.export_to_markdown()
except Exception as exc:
    conversion_error=f"{type(exc).__name__}: {exc}"
    print("DocTags→Markdown 转换未完成，已保留原始 DocTags：",conversion_error)
(OUTPUT_DIR/"result.dt").write_text(doctags,encoding="utf-8")
if result_md is not None: (OUTPUT_DIR/"result.md").write_text(result_md,encoding="utf-8")
meta={
    "model_id":MODEL_ID,
    "model_role":"public_baseline_not_lab_checkpoint",
    "device":torch.cuda.get_device_name(0),
    "torch_version":torch.__version__,
    "input_file":image_path.name,
    "input_sha256":hashlib.sha256(image_path.read_bytes()).hexdigest(),
    "image_size":list(image.size),
    "max_new_tokens":MAX_NEW_TOKENS,
    "generated_tokens":int(new_tokens.shape[1]),
    "model_load_seconds":round(load_seconds,2),
    "inference_seconds":round(infer_seconds,2),
    "doctags_chars":len(doctags),
    "markdown_conversion_ok":result_md is not None,
    "markdown_conversion_error":conversion_error,
}
(OUTPUT_DIR/"run_metadata.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps({k:meta[k] for k in ["model_id","generated_tokens","inference_seconds","doctags_chars","markdown_conversion_ok"]},ensure_ascii=False,indent=2))
print("输出目录:",OUTPUT_DIR)
print("文件:",[p.name for p in OUTPUT_DIR.iterdir()])
if result_md: print("\nMarkdown 预览:\n",result_md[:1000])

## 7. 学生验收
运行成功后，在 Kaggle 的 **Output** 下载 `input_page.png`、`result.dt`、`run_metadata.json`，如果生成成功也下载 `result.md`。把 GPU 型号、运行时间、是否输出有效 DocTags 反馈给老师。

**注意：** 请不要将包含私人文档内容的 Notebook 保存为 Public；公开发布只保留合成或获得授权的样例。结束后停止 GPU Session，避免消耗免费配额。